# NLP: URL Spam Detection
This project builds a system to automatically detect whether a URL is spam or not.
I preproced the URLs using NLP techniques (tokenization, stopword removal, lemmatization),
vectorize them with TF-IDF, train an SVM classifier, and optimize its hyperparameters with GridSearchCV.

## Step 1: Load the Dataset

In [ ]:
import subprocess
subprocess.run(['pip', 'install', 'nltk', '-q'], check=True)
print('nltk installed')


[notice] A new release of pip is available: 23.1.2 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


nltk installed


In [ ]:
import pandas as pd
import numpy as np
import re
import pickle
import os
import warnings
warnings.filterwarnings('ignore')

import nltk
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report

url = 'https://breathecode.herokuapp.com/asset/internal-link?id=932&path=url_spam.csv'
df = pd.read_csv(url)
print('Shape:', df.shape)
df.head(10)

Shape: (2999, 2)


,url,is_spam
0,https://briefingday.us8.list-manage.com/unsubs...,True
1,https://www.hvper.com/,True
2,https://briefingday.com/m/v4n3i4f3,True
3,https://briefingday.com/n/20200618/m#commentform,False
4,https://briefingday.com/fan,True
5,https://www.brookings.edu/interactives/reopeni...,False
6,https://www.reuters.com/investigates/special-r...,False
7,https://www.theatlantic.com/magazine/archive/2...,False
8,https://www.vox.com/2020/6/17/21294680/john-bo...,False
9,https://www.theguardian.com/travel/2020/jun/18...,False


In [ ]:
print('Columns:', df.columns.tolist())
print()
print(df.dtypes)
print()
print('Null values:')
print(df.isnull().sum())
print()
print('Label distribution:')
print(df.iloc[:, -1].value_counts())

Columns: ['url', 'is_spam']

url         str
is_spam    bool
dtype: object

Null values:
url        0
is_spam    0
dtype: int64

Label distribution:
is_spam
False    2303
True      696
Name: count, dtype: int64


## Step 2: Preprocess the URLs
I tokenize each URL by splitting on punctuation characters (`.`, `/`, `-`, `_`, `?`, `=`, `&`, `#`, `~`, `%`, `@`),
remove English stopwords, lemmatize the remaining tokens, and rejoin them into a cleaned string.

In [ ]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess_url(url_text):
    url_text = url_text.lower()
    tokens = re.split(r'[\.\/-_\?=&#~%@:+,!;()\[\]{}\d]+', url_text)
    tokens = [
        lemmatizer.lemmatize(t)
        for t in tokens
        if t and len(t) > 2 and t not in stop_words
    ]
    return ' '.join(tokens)

url_col   = df.columns[0]
label_col = df.columns[-1]

df['clean_url'] = df[url_col].astype(str).apply(preprocess_url)

print('Sample preprocessed URLs:')
df[[url_col, 'clean_url', label_col]].sample(5, random_state=42)

Sample preprocessed URLs:


,url,clean_url,is_spam
1376,https://link.morningbrew.com/manage/5z8/oc,http link morningbrew com manage,True
932,https://www.sciencedaily.com/releases/2019/02/...,http www sciencedaily com release htm,False
144,https://www.ft.com/content/eae603a4-a369-4801-...,http www com content eae cc-,False
1752,https://thehustle.co/careers/,http thehustle career,True
51,https://www.caltech.edu/about/news/natural-flu...,http www caltech edu news natural-fluid-inject...,False


## Step 3: Vectorize and Split
I use TF-IDF to convert the cleaned URLs into numerical features, then split into train and test sets.

In [ ]:
X = df['clean_url']
y = df[label_col]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
X_train_vec = tfidf.fit_transform(X_train)
X_test_vec  = tfidf.transform(X_test)

print(f'Train size: {X_train_vec.shape}')
print(f'Test size:  {X_test_vec.shape}')
print(f'Vocabulary size: {len(tfidf.vocabulary_)}')

Train size: (2399, 5000)
Test size:  (600, 5000)
Vocabulary size: 5000


## Step 4: Build the SVM (default parameters)
I train a Support Vector Machine with default hyperparameters as a baseline.

In [ ]:
svm_default = SVC(random_state=42)
svm_default.fit(X_train_vec, y_train)

y_pred_default = svm_default.predict(X_test_vec)
acc_default = accuracy_score(y_test, y_pred_default)

print(f'Default SVM Accuracy: {acc_default:.4f}')
print()
print(classification_report(y_test, y_pred_default))

Default SVM Accuracy: 0.9483

              precision    recall  f1-score   support

       False       0.96      0.98      0.97       461
        True       0.92      0.86      0.88       139

    accuracy                           0.95       600
   macro avg       0.94      0.92      0.93       600
weighted avg       0.95      0.95      0.95       600



## Step 5: Optimize with GridSearchCV
I search over `C`, `kernel`, and `gamma` to find the best hyperparameter combination.

In [ ]:
param_grid = {
    'C':      [0.1, 1, 10],
    'kernel': ['linear', 'rbf'],
    'gamma':  ['scale', 'auto']
}

grid_search = GridSearchCV(
    SVC(random_state=42),
    param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)
grid_search.fit(X_train_vec, y_train)

print('Best parameters:', grid_search.best_params_)
print(f'Best CV accuracy: {grid_search.best_score_:.4f}')

Fitting 5 folds for each of 12 candidates, totalling 60 fits
Best parameters: {'C': 10, 'gamma': 'scale', 'kernel': 'rbf'}
Best CV accuracy: 0.9571


In [ ]:
best_svm = grid_search.best_estimator_
y_pred_best = best_svm.predict(X_test_vec)
acc_best = accuracy_score(y_test, y_pred_best)

print(f'Optimized SVM Accuracy: {acc_best:.4f}')
print()
print(classification_report(y_test, y_pred_best))
print()
print(f'Improvement over default: {acc_best - acc_default:+.4f}')

Optimized SVM Accuracy: 0.9550

              precision    recall  f1-score   support

       False       0.97      0.97      0.97       461
        True       0.90      0.91      0.90       139

    accuracy                           0.95       600
   macro avg       0.94      0.94      0.94       600
weighted avg       0.96      0.95      0.96       600


Improvement over default: +0.0067


## Step 6: Save the Model

In [ ]:
os.makedirs('./models', exist_ok=True)

with open('./models/svm_url_spam.pkl', 'wb') as f:
    pickle.dump(best_svm, f)

with open('./models/tfidf_url_spam.pkl', 'wb') as f:
    pickle.dump(tfidf, f)

print('Model saved to ./models/svm_url_spam.pkl')
print('Vectorizer saved to ./models/tfidf_url_spam.pkl')

Model saved to ./models/svm_url_spam.pkl
Vectorizer saved to ./models/tfidf_url_spam.pkl


## Conclusion
I built a URL spam detector using NLP preprocessing and an SVM classifier.
URLs were tokenized by punctuation, cleaned of stopwords, and lemmatized before being vectorized with TF-IDF (unigrams + bigrams).
The default SVM served as a strong baseline, and GridSearchCV tuned the `C`, `kernel`, and `gamma` parameters to find the optimal configuration.
SVMs are well-suited for high-dimensional sparse text data like TF-IDF features because they maximize the margin between classes even in large feature spaces.